# 训练 · 打分 · 回测 Pipeline（XGB + Transformer + Ensemble）

- **与 `main.ipynb` 的关系**：二者都合理——`main.ipynb` 更接近「全历史滚动、默认落盘到 `outputs/scores/`」的**规范实盘前**流程；本 notebook 侧重 **OOS 严格冻结 + 测试对比**（模型与打分写入 `config.PIPELINE_HOLDOUT_*`，不会覆盖 main 的 `SCORE_xgb_*.fea` / `rolling_model.pkl` 等）。
- **训练**：仅 `train ∪ val`（`TRADE_DATE < OOS_START`），与 [main.ipynb](main.ipynb) 中纯 OOS 冻结思路一致。
- **分别输出**：XGB 与 Transformer 各自的 fold IC、全样本 IC 报告、快速分层回测图（含组内累计收益曲线）。
- **Ensemble**：对两路 `full` 打分（样本内+OOS 拼接）做截面 Z-Score 后简单平均，再跑同样报告（实现见 [ensemble_scorer.py](model/ensemble_scorer.py)，与 main 中用法一致）。
- **精细化回测**：可对三份 `score_full` 分别跑 `BacktestRunner`（可选）。

In [1]:
import sys
import logging
from pathlib import Path

ROOT = Path("/root/autodl-tmp")
sys.path.insert(0, str(ROOT))

logging.basicConfig(level=logging.INFO, force=True)
for name in ("Strategy", "Strategy.model", "Strategy.backtest"):
    logging.getLogger(name).setLevel(logging.INFO)

import pandas as pd
import numpy as np

from Strategy import config

## 参数

- `TRAIN_TRANSFORMER`：若只想先跑通 XGB，可改为 `False`。
- **产出目录**：`config.PIPELINE_HOLDOUT_SCORE_DIR`（滚动模型 pkl、各 `SCORE_*.fea`）、`COMPARE_ROOT`（本格基于 `PIPELINE_HOLDOUT_BT_DIR` 生成，放 IC/quick/事件回测图）。与 main 默认的 `SCORE_OUTPUT_DIR` 根目录隔离。
- `BT_PRICE_TAG`：与 `outputs/labels/{BT_PRICE_TAG}.fea` 成交价宽表一致。

In [2]:
LABEL_TAG = "OPEN0935_1000"
BT_PRICE_TAG = "OPEN0935_1000"

TRAIN_TRANSFORMER = True

config.PIPELINE_HOLDOUT_SCORE_DIR.mkdir(parents=True, exist_ok=True)
config.PIPELINE_HOLDOUT_BT_DIR.mkdir(parents=True, exist_ok=True)

COMPARE_ROOT = config.PIPELINE_HOLDOUT_BT_DIR / f"compare_{LABEL_TAG}"
COMPARE_ROOT.mkdir(parents=True, exist_ok=True)

TOP_KS = (20, 50, 100)
TAIL_KS = (20, 50, 100)

print("OOS_START =", config.OOS_START)
print("Pipeline 打分目录:", config.PIPELINE_HOLDOUT_SCORE_DIR)
print("对比报告目录:", COMPARE_ROOT)

OOS_START = 2024-09-01
Pipeline 打分目录: /root/autodl-tmp/Strategy/outputs/scores/pipeline_holdout
对比报告目录: /root/autodl-tmp/Strategy/outputs/bt_results/pipeline_holdout/compare_OPEN0935_1000


## 1. 数据：全量 Panel + 切分（训练不含 OOS）

In [3]:
from Strategy.factor.factor_base import load_all_factors
from Strategy.label.label_generator import load_label
from Strategy.model.trainer import build_panel, split_panel

factor_dict = load_all_factors()
label_df = load_label(LABEL_TAG)

panel = build_panel(factor_dict, label_df)
train, val, oos = split_panel(panel)

panel_in_sample = pd.concat([train, val], axis=0, ignore_index=True)
panel_in_sample = panel_in_sample.sort_values(["TRADE_DATE", "StockID"]).reset_index(drop=True)

oos_start = pd.Timestamp(config.OOS_START).normalize()
assert pd.to_datetime(panel_in_sample["TRADE_DATE"]).max().normalize() < oos_start
assert pd.to_datetime(oos["TRADE_DATE"]).min().normalize() >= oos_start

print("panel:", panel.shape, "| in_sample:", panel_in_sample.shape, "| oos:", oos.shape)

INFO:Strategy.model.trainer:Panel 对齐: 1251 交易日 × 5324 只股票, 134 个因子
INFO:Strategy.model.trainer:正在 concat 135 个 Series ...
INFO:Strategy.model.trainer:Panel 构建完成: shape=(6660324, 137)
INFO:Strategy.model.trainer:panel 日期: [2021-01-18, 2026-03-20]
INFO:Strategy.model.trainer:Train: 3391388 rows, Val: 1288408 rows, OOS: 1980528 rows


panel: (6660324, 137) | in_sample: (4679796, 137) | oos: (1980528, 137)


## 2a. XGBoost 滚动训练（仅 in-sample）

In [4]:
from Strategy.model.rolling_trainer import RollingTrainer
from Strategy.model.trainer import XGBTrainer

rt_xgb = RollingTrainer(
    model_class=XGBTrainer,
    model_kwargs={"use_wpcc": True, "label_winsorize_sigma": 0},
    train_start="2021-01-01",
    first_train_months=9,
    val_months=3,
)
rt_xgb.train_all_folds(panel_in_sample)

print("\n========== XGB fold IC ==========")
print(rt_xgb.fold_ic_report())
fi = rt_xgb.get_feature_importance()
if fi is not None:
    print(fi.head(10))

rt_xgb.save_model(config.PIPELINE_HOLDOUT_SCORE_DIR / f"rolling_xgb_{LABEL_TAG}.pkl")

INFO:Strategy.model.rolling_trainer:滚动训练: 12 个 Fold, 特征数=134
INFO:Strategy.model.rolling_trainer:━━ Fold 1: Train=[2021-01-01, 2021-09-30] Val=[2021-10-01, 2021-12-31] ━━


INFO:Strategy.model.rolling_trainer:  Train: 915728 rows, Val: 324764 rows
INFO:Strategy.model.trainer:特征数=134
INFO:Strategy.model.trainer:训练样本 n=740311; label std=0.0359565; 特征 NaN=0.24%
INFO:Strategy.model.trainer:使用 WPCC 自定义目标函数


[0]	train-rmse:0.49967	train-wpcc:-0.07112	val-rmse:0.49920	val-wpcc:-0.02168
[96]	train-rmse:0.49967	train-wpcc:-0.21971	val-rmse:0.49919	val-wpcc:-0.00857


INFO:Strategy.model.trainer:训练完成. best_iteration=16
INFO:Strategy.model.rolling_trainer:  Fold 1 Val IC: mean=0.0086  std=0.0386  ICIR=0.2222  RankIC=0.0031  days=61
INFO:Strategy.model.rolling_trainer:━━ Fold 2: Train=[2021-01-01, 2021-12-31] Val=[2022-01-01, 2022-03-31] ━━
INFO:Strategy.model.rolling_trainer:  Train: 1240492 rows, Val: 308792 rows
INFO:Strategy.model.trainer:特征数=134
INFO:Strategy.model.trainer:训练样本 n=1017104; label std=0.0354596; 特征 NaN=0.22%
INFO:Strategy.model.trainer:使用 WPCC 自定义目标函数


[0]	train-rmse:0.49954	train-wpcc:-0.06246	val-rmse:0.50303	val-wpcc:-0.03215
[88]	train-rmse:0.49954	train-wpcc:-0.19134	val-rmse:0.50303	val-wpcc:-0.03333


...

INFO:Strategy.model.trainer:训练完成. best_iteration=33
INFO:Strategy.model.rolling_trainer:  Fold 11 Val IC: mean=0.0549  std=0.0688  ICIR=0.7979  RankIC=0.0155  days=59
INFO:Strategy.model.rolling_trainer:━━ Fold 12: Train=[2021-01-01, 2024-06-30] Val=[2024-07-01, 2024-09-30] ━━
INFO:Strategy.model.rolling_trainer:  Train: 4440216 rows, Val: 239580 rows
INFO:Strategy.model.trainer:特征数=134
INFO:Strategy.model.trainer:训练样本 n=3968314; label std=0.0342547; 特征 NaN=0.14%
INFO:Strategy.model.trainer:使用 WPCC 自定义目标函数


[0]	train-rmse:0.50061	train-wpcc:-0.04715	val-rmse:0.50061	val-wpcc:-0.04822
[81]	train-rmse:0.50061	train-wpcc:-0.13033	val-rmse:0.50062	val-wpcc:-0.05193


INFO:Strategy.model.trainer:训练完成. best_iteration=1
INFO:Strategy.model.rolling_trainer:  Fold 12 Val IC: mean=0.0519  std=0.0544  ICIR=0.9540  RankIC=0.0265  days=45
INFO:Strategy.model.rolling_trainer:滚动训练完成: 12 个 Fold
INFO:Strategy.model.rolling_trainer:Feature importance 汇总: 134 个特征, top5: ['vol_parkinson_d', 'ar_7', 'ma5', 'avg_turnover_20d', 'gap_ratio']
INFO:Strategy.model.rolling_trainer:滚动模型已保存: /root/autodl-tmp/Strategy/outputs/scores/pipeline_holdout/rolling_xgb_OPEN0935_1000.pkl (12 folds)



========== XGB fold IC ==========
    fold_id status  val_ic_mean  val_ic_std  val_icir  val_rank_ic_mean  \
0         1     ok     0.008575    0.038593  0.222195          0.003055   
1         2     ok     0.033328    0.047009  0.708961          0.025862   
2         3     ok     0.031255    0.051094  0.611709          0.019317   
3         4     ok     0.036843    0.043120  0.854443          0.033392   
4         5     ok     0.030928    0.046466  0.665613          0.031591   
5         6     ok     0.032574    0.054020  0.602996          0.032840   
6         7     ok     0.020260    0.047077  0.430366         -0.002166   
7         8     ok     0.026590    0.050513  0.526396          0.022758   
8         9     ok     0.032609    0.063728  0.511699          0.043344   
9        10     ok     0.026224    0.098983  0.264935          0.019024   
10       11     ok     0.054926    0.068840  0.797885          0.015550   
11       12     ok     0.051930    0.054434  0.954004          0.

PosixPath('/root/autodl-tmp/Strategy/outputs/scores/pipeline_holdout/rolling_xgb_OPEN0935_1000.pkl')

## 2b. Transformer 滚动训练（仅 in-sample）

关闭可将 `TRAIN_TRANSFORMER=False`，后续 ensemble 将跳过（仅保留单模型报告）。

In [5]:
rt_tf = None
if TRAIN_TRANSFORMER:
    from Strategy.model.transformer_trainer import TransformerTrainer

    rt_tf = RollingTrainer(
        model_class=TransformerTrainer,
        model_kwargs={
            "d_model": 64, "nhead": 4, "num_layers": 2, "d_ff": 128, "dropout": 0.15,
            "epochs": 50, "lr": 5e-4, "weight_decay": 0.01, "batch_size": 16,
            "early_stopping_patience": 8, "device": "cuda", "use_amp": False,
            "label_winsorize_sigma": 0,
        },
        train_start="2021-01-01",
        first_train_months=9,
        val_months=3,
    )
    rt_tf.train_all_folds(panel_in_sample)
    print("\n========== Transformer fold IC ==========")
    print(rt_tf.fold_ic_report())
    rt_tf.save_model(config.PIPELINE_HOLDOUT_SCORE_DIR / f"rolling_transformer_{LABEL_TAG}.pkl")
else:
    print("已跳过 Transformer 训练 (TRAIN_TRANSFORMER=False)")

INFO:Strategy.model.rolling_trainer:滚动训练: 12 个 Fold, 特征数=134
INFO:Strategy.model.rolling_trainer:━━ Fold 1: Train=[2021-01-01, 2021-09-30] Val=[2021-10-01, 2021-12-31] ━━
INFO:Strategy.model.rolling_trainer:  Train: 915728 rows, Val: 324764 rows
INFO:Strategy.model.transformer_trainer:训练设备: cuda, AMP: False
INFO:Strategy.model.transformer_trainer:CrossSectionalTransformer: d_input=134, d_model=64, nhead=4, num_layers=2, d_ff=128, dropout=0.15, params=0.08M
INFO:Strategy.model.transformer_trainer:训练集: 172 日, 验证集: 61 日, 特征: 134
INFO:Strategy.model.transformer_trainer:  Epoch 1/50  train=-0.00588  val=-0.00101  lr=5.00e-04
INFO:Strategy.model.transformer_trainer:  Epoch 2/50  train=-0.01306  val=-0.00201  lr=4.98e-04
INFO:Strategy.model.transformer_trainer:  Epoch 3/50  train=-0.01496  val=-0.00185  lr=4.96e-04
INFO:Strategy.model.transformer_trainer:  Epoch 5/50  train=-0.01597  val=-0.00133  lr=4.88e-04
INFO:Strategy.model.transformer_trainer:  Epoch 10/50  train=-0.01959  val=0.00020  


========== Transformer fold IC ==========
    fold_id status  val_ic_mean  val_ic_std  val_icir  val_rank_ic_mean  \
0         1     ok     0.003701    0.075674  0.048912         -0.001393   
1         2     ok     0.022945    0.096413  0.237992          0.012435   
2         3     ok     0.018841    0.069292  0.271910          0.010198   
3         4     ok     0.009411    0.072612  0.129603         -0.003901   
4         5     ok     0.010155    0.051812  0.195998          0.003171   
5         6     ok     0.007990    0.057278  0.139487          0.001713   
6         7     ok    -0.014775    0.077549 -0.190519         -0.024858   
7         8     ok     0.004038    0.070088  0.057616          0.007936   
8         9     ok     0.008174    0.081431  0.100384          0.019272   
9        10     ok     0.013965    0.115755  0.120644          0.011330   
10       11     ok     0.014021    0.056247  0.249280          0.015397   
11       12     ok     0.029346    0.051246  0.572655    

## 3. 打分：各模型 in-sample / OOS / full，并落盘

`full` = 样本内与 OOS 按日期拼接；文件落在 **`PIPELINE_HOLDOUT_SCORE_DIR`**，不会与 main 根目录下的 `load_scores("xgb", ...)` 默认路径冲突。读取时用 `pd.read_feather(...).set_index("TRADE_DATE")` 或自行封装路径。

In [6]:
from Strategy.model.scorer import mask_scores_by_current_price
from Strategy.data_io.saver import save_wide_table

def _score_and_save(rt, tag: str):
    s_is = mask_scores_by_current_price(rt.score_all(panel_in_sample, normalize=True), label_tag=LABEL_TAG)
    s_oos = mask_scores_by_current_price(rt.score_all(oos, normalize=True), label_tag=LABEL_TAG)
    s_full = pd.concat([s_is, s_oos]).sort_index()
    s_full = s_full[~s_full.index.duplicated(keep="last")]
    d = config.PIPELINE_HOLDOUT_SCORE_DIR
    save_wide_table(s_is, d / f"SCORE_{tag}_{LABEL_TAG}_in_sample.fea")
    save_wide_table(s_oos, d / f"SCORE_{tag}_{LABEL_TAG}_oos.fea")
    save_wide_table(s_full, d / f"SCORE_{tag}_{LABEL_TAG}.fea")
    return s_is, s_oos, s_full

score_xgb_is, score_xgb_oos, score_xgb_full = _score_and_save(rt_xgb, "xgb")
print("XGB shapes:", score_xgb_is.shape, score_xgb_oos.shape, score_xgb_full.shape)

score_tf_is = score_tf_oos = score_tf_full = None
if rt_tf is not None:
    score_tf_is, score_tf_oos, score_tf_full = _score_and_save(rt_tf, "transformer")
    print("Transformer shapes:", score_tf_is.shape, score_tf_oos.shape, score_tf_full.shape)

INFO:Strategy.model.scorer:score 已按 T 日价格 mask: before=4679796 after=4198806 removed=480990
INFO:Strategy.model.scorer:score 已按 T 日价格 mask: before=1980528 after=1904717 removed=75811


XGB shapes: (879, 5324) (372, 5324) (1251, 5324)


INFO:Strategy.model.scorer:score 已按 T 日价格 mask: before=4679796 after=4198806 removed=480990
INFO:Strategy.model.scorer:score 已按 T 日价格 mask: before=1980528 after=1904717 removed=75811


Transformer shapes: (879, 5324) (372, 5324) (1251, 5324)


## 4. Ensemble 打分（Z-Score 简单平均）

逻辑与 main 中 `ensemble_scorer` 相同；落盘为 **`PIPELINE_HOLDOUT_SCORE_DIR / SCORE_ensemble_{LABEL_TAG}.fea`**（main 默认仍写 `SCORE_OUTPUT_DIR`，二者互不覆盖）。

In [7]:
from Strategy.model.ensemble_scorer import compute_score_correlation, select_ensemble_models, ensemble_scores

score_ensemble_full = None
if rt_tf is not None:
    dfs = {"xgb": score_xgb_full, "transformer": score_tf_full}
    print("\n========== 模型打分截面平均相关性 ==========")
    print(compute_score_correlation(dfs))
    selected = select_ensemble_models(dfs, ic_summaries=None, max_pairwise_corr=0.95)
    print("入选集成模型:", selected)
    score_ensemble_full = ensemble_scores(
        dfs, selected_models=selected, label_tag=LABEL_TAG, save=False
    )
    score_ensemble_full = mask_scores_by_current_price(score_ensemble_full, label_tag=LABEL_TAG)
    save_wide_table(score_ensemble_full, config.PIPELINE_HOLDOUT_SCORE_DIR / f"SCORE_ensemble_{LABEL_TAG}.fea")
    print("ensemble full:", score_ensemble_full.shape)
else:
    print("无 Transformer 打分，跳过 ensemble")


========== 模型打分截面平均相关性 ==========


INFO:Strategy.model.ensemble_scorer:模型打分相关性矩阵 (1251 日平均):
             transformer      xgb
transformer      1.00000  0.06286
xgb              0.06286  1.00000


             transformer      xgb
transformer      1.00000  0.06286
xgb              0.06286  1.00000


INFO:Strategy.model.ensemble_scorer:模型打分相关性矩阵 (1251 日平均):
             transformer      xgb
transformer      1.00000  0.06286
xgb              0.06286  1.00000
INFO:Strategy.model.ensemble_scorer:集成模型入选: ['xgb', 'transformer'] (共 2 / 2)
INFO:Strategy.model.ensemble_scorer:集成 2 个模型: ['xgb', 'transformer']


入选集成模型: ['xgb', 'transformer']


/root/autodl-tmp/Strategy/model/ensemble_scorer.py:246: RuntimeWarning: Mean of empty slice
  avg_scores = np.nanmean(stacked, axis=0)                   # (n_dates, n_stocks)
INFO:Strategy.model.ensemble_scorer:集成打分: 1251 dates × 5324 stocks
INFO:Strategy.model.scorer:score 已按 T 日价格 mask: before=6103523 after=6103523 removed=0


ensemble full: (1251, 5324)


## 5. 对比报告：IC / Rank IC（分段统计 + 累计 IC 图）+ 快速分层（累计收益曲线）

对 **xgb / transformer / ensemble** 各输出一套子目录（与 main 中 `models_to_test` 循环类似）：

- `ic_analysis.png`、`ic_series.csv`、`ic_summary.csv` —— [ic_analysis.run_ic_analysis](backtest/ic_analysis.py)
- `quantile_backtest.png`、`topN_tailN_nav.png` —— [quick_backtest.run_quick_backtest](backtest/quick_backtest.py)（`run_ic=False` 避免与上重复）

区间：`VAL_START` 起至数据末（含纯 OOS）。另可对 **仅 OOS** 再跑一版 quick（见下一格）。

In [8]:
from Strategy.backtest.ic_analysis import run_ic_analysis
from Strategy.backtest.quick_backtest import run_quick_backtest

label_wide = load_label(LABEL_TAG)

def report_one(name: str, score_full: pd.DataFrame):
    if score_full is None or len(score_full) == 0:
        print(f"跳过 {name}: 无打分")
        return
    base = COMPARE_ROOT / name
    base.mkdir(parents=True, exist_ok=True)
    print(f"\n{'='*20} {name} | IC 分析 {'='*20}")
    ic_dir = base / "ic_val_to_end"
    ic_df, summary = run_ic_analysis(
        score_full,
        label_wide,
        title=f"{name.upper()} | {LABEL_TAG} | IC",
        output_dir=ic_dir,
        rolling_window=20,
    )
    print(summary)
    print(f"\n--- {name} | quick 分层 (VAL->末, 含OOS) ---")
    run_quick_backtest(
        score_full,
        label_wide,
        n_groups=config.N_QUANTILE_GROUPS,
        title=f"{name.upper()} | {LABEL_TAG} | quick",
        output_dir=base / "quick_val_to_end",
        start_date=config.VAL_START,
        end_date=None,
        top_ks=TOP_KS,
        tail_ks=TAIL_KS,
        twap_tag=BT_PRICE_TAG,
        run_ic=False,
    )

report_one("xgb", score_xgb_full)
report_one("transformer", score_tf_full)
report_one("ensemble", score_ensemble_full)


==================== xgb | IC 分析 ====================


INFO:Strategy.backtest.ic_analysis:  [IC][Train] mean=0.0796  std=0.0987  ICIR=0.8065  IC_WinRate=81.00%  n=637
INFO:Strategy.backtest.ic_analysis:  [IC][Val] mean=0.0393  std=0.0744  ICIR=0.5288  IC_WinRate=76.86%  n=242
INFO:Strategy.backtest.ic_analysis:  [IC][OOS] mean=0.0316  std=0.0561  ICIR=0.5628  IC_WinRate=70.62%  n=371
INFO:Strategy.backtest.ic_analysis:  [IC][Overall] mean=0.0575  std=0.0864  ICIR=0.6657  IC_WinRate=77.12%  n=1250
INFO:Strategy.backtest.ic_analysis:  [RANK_IC][Train] mean=0.0475  std=0.0820  ICIR=0.5795  IC_WinRate=68.13%  n=637
INFO:Strategy.backtest.ic_analysis:  [RANK_IC][Val] mean=0.0241  std=0.1367  ICIR=0.1761  IC_WinRate=58.26%  n=242
INFO:Strategy.backtest.ic_analysis:  [RANK_IC][OOS] mean=0.0293  std=0.1023  ICIR=0.2862  IC_WinRate=59.30%  n=371
INFO:Strategy.backtest.ic_analysis:  [RANK_IC][Overall] mean=0.0376  std=0.1012  ICIR=0.3714  IC_WinRate=63.60%  n=1250
INFO:Strategy.backtest.ic_analysis:IC 时间序列已保存: /root/autodl-tmp/Strategy/outputs/bt_re

                 n_days      mean       std      icir  ic_win_rate
metric  segment                                                   
ic      Train       637  0.079591  0.098690  0.806469     0.810047
        Val         242  0.039319  0.074359  0.528772     0.768595
        OOS         371  0.031551  0.056060  0.562795     0.706199
        Overall    1250  0.057536  0.086432  0.665680     0.771200
rank_ic Train       637  0.047529  0.082014  0.579521     0.681319
        Val         242  0.024058  0.136655  0.176050     0.582645
        OOS         371  0.029274  0.102267  0.286247     0.592992
        Overall    1250  0.037567  0.101158  0.371365     0.636000

--- xgb | quick 分层 (VAL->末, 含OOS) ---


INFO:Strategy.backtest.quick_backtest:分层回测完成: 613 交易日, 20 组, topK=(20, 50, 100), tailK=(20, 50, 100)
INFO:Strategy.backtest.quick_backtest:stock pool: score=5107 tradeable=3031 no_twap=0 limit_up=12 new=4 delist=0 st=590 prefix=1467
INFO:Strategy.backtest.quick_backtest:  group1: Ann=30.88% MDD=-44.68% SR=0.86 WR=53.18%
INFO:Strategy.backtest.quick_backtest:  group2: Ann=46.12% MDD=-44.82% SR=1.41 WR=56.93%
INFO:Strategy.backtest.quick_backtest:  group3: Ann=46.30% MDD=-42.75% SR=1.47 WR=57.91%
INFO:Strategy.backtest.quick_backtest:  group4: Ann=42.77% MDD=-41.16% SR=1.38 WR=56.77%
INFO:Strategy.backtest.quick_backtest:  group5: Ann=41.99% MDD=-41.99% SR=1.43 WR=56.93%
INFO:Strategy.backtest.quick_backtest:  group6: Ann=38.46% MDD=-40.24% SR=1.33 WR=57.26%
INFO:Strategy.backtest.quick_backtest:  group7: Ann=37.79% MDD=-40.31% SR=1.33 WR=56.61%
INFO:Strategy.backtest.quick_backtest:  group8: Ann=34.10% MDD=-39.90% SR=1.21 WR=56.28%
INFO:Strategy.backtest.quick_backtest:  group9: Ann=35.


==================== transformer | IC 分析 ====================


...

INFO:Strategy.backtest.quick_backtest:分层回测完成: 613 交易日, 20 组, topK=(20, 50, 100), tailK=(20, 50, 100)
INFO:Strategy.backtest.quick_backtest:stock pool: score=5107 tradeable=3031 no_twap=0 limit_up=12 new=4 delist=0 st=590 prefix=1467
INFO:Strategy.backtest.quick_backtest:  group1: Ann=26.35% MDD=-42.73% SR=0.85 WR=53.18%
INFO:Strategy.backtest.quick_backtest:  group2: Ann=35.39% MDD=-42.49% SR=1.20 WR=55.79%
INFO:Strategy.backtest.quick_backtest:  group3: Ann=39.71% MDD=-40.36% SR=1.36 WR=55.63%
INFO:Strategy.backtest.quick_backtest:  group4: Ann=36.78% MDD=-38.93% SR=1.28 WR=57.42%
INFO:Strategy.backtest.quick_backtest:  group5: Ann=39.25% MDD=-37.27% SR=1.38 WR=57.10%
INFO:Strategy.backtest.quick_backtest:  group6: Ann=31.76% MDD=-38.44% SR=1.13 WR=56.61%
INFO:Strategy.backtest.quick_backtest:  group7: Ann=36.55% MDD=-38.53% SR=1.30 WR=57.59%
INFO:Strategy.backtest.quick_backtest:  group8: Ann=35.68% MDD=-38.46% SR=1.28 WR=57.59%
INFO:Strategy.backtest.quick_backtest:  group9: Ann=32.


==================== ensemble | IC 分析 ====================


INFO:Strategy.backtest.ic_analysis:  [IC][Train] mean=0.0490  std=0.0814  ICIR=0.6017  IC_WinRate=72.68%  n=637
INFO:Strategy.backtest.ic_analysis:  [IC][Val] mean=0.0313  std=0.0875  ICIR=0.3580  IC_WinRate=67.77%  n=242
INFO:Strategy.backtest.ic_analysis:  [IC][OOS] mean=0.0244  std=0.0780  ICIR=0.3125  IC_WinRate=62.26%  n=371
INFO:Strategy.backtest.ic_analysis:  [IC][Overall] mean=0.0383  std=0.0824  ICIR=0.4647  IC_WinRate=68.64%  n=1250
INFO:Strategy.backtest.ic_analysis:  [RANK_IC][Train] mean=0.0268  std=0.0828  ICIR=0.3241  IC_WinRate=64.52%  n=637
INFO:Strategy.backtest.ic_analysis:  [RANK_IC][Val] mean=0.0252  std=0.1079  ICIR=0.2333  IC_WinRate=58.68%  n=242
INFO:Strategy.backtest.ic_analysis:  [RANK_IC][OOS] mean=0.0298  std=0.0910  ICIR=0.3270  IC_WinRate=60.65%  n=371
INFO:Strategy.backtest.ic_analysis:  [RANK_IC][Overall] mean=0.0274  std=0.0905  ICIR=0.3025  IC_WinRate=62.24%  n=1250
INFO:Strategy.backtest.ic_analysis:IC 时间序列已保存: /root/autodl-tmp/Strategy/outputs/bt_re

                 n_days      mean       std      icir  ic_win_rate
metric  segment                                                   
ic      Train       637  0.048997  0.081430  0.601701     0.726845
        Val         242  0.031322  0.087490  0.358013     0.677686
        OOS         371  0.024386  0.078047  0.312457     0.622642
        Overall    1250  0.038271  0.082360  0.464673     0.686400
rank_ic Train       637  0.026822  0.082767  0.324065     0.645212
        Val         242  0.025162  0.107865  0.233270     0.586777
        OOS         371  0.029769  0.091031  0.327017     0.606469
        Overall    1250  0.027375  0.090503  0.302475     0.622400

--- ensemble | quick 分层 (VAL->末, 含OOS) ---


INFO:Strategy.backtest.quick_backtest:分层回测完成: 613 交易日, 20 组, topK=(20, 50, 100), tailK=(20, 50, 100)
INFO:Strategy.backtest.quick_backtest:stock pool: score=5107 tradeable=3031 no_twap=0 limit_up=12 new=4 delist=0 st=590 prefix=1467
INFO:Strategy.backtest.quick_backtest:  group1: Ann=34.64% MDD=-46.27% SR=1.01 WR=53.34%
INFO:Strategy.backtest.quick_backtest:  group2: Ann=44.42% MDD=-45.43% SR=1.42 WR=57.75%
INFO:Strategy.backtest.quick_backtest:  group3: Ann=39.25% MDD=-43.99% SR=1.31 WR=56.28%
INFO:Strategy.backtest.quick_backtest:  group4: Ann=42.63% MDD=-41.31% SR=1.46 WR=57.59%
INFO:Strategy.backtest.quick_backtest:  group5: Ann=42.88% MDD=-41.28% SR=1.48 WR=57.42%
INFO:Strategy.backtest.quick_backtest:  group6: Ann=41.00% MDD=-36.74% SR=1.44 WR=56.61%
INFO:Strategy.backtest.quick_backtest:  group7: Ann=41.06% MDD=-37.20% SR=1.47 WR=58.56%
INFO:Strategy.backtest.quick_backtest:  group8: Ann=31.01% MDD=-37.19% SR=1.14 WR=55.14%
INFO:Strategy.backtest.quick_backtest:  group9: Ann=35.

## 5b. 仅纯 OOS 区间：快速分层（便于对照样本外累计收益）

In [9]:
def quick_oos_only(name: str, score_oos: pd.DataFrame):
    if score_oos is None or len(score_oos) == 0:
        return
    out = COMPARE_ROOT / name / "quick_oos_only"
    run_quick_backtest(
        score_oos,
        label_wide,
        n_groups=config.N_QUANTILE_GROUPS,
        title=f"{name.upper()} | {LABEL_TAG} | OOS",
        output_dir=out,
        start_date=config.OOS_START,
        end_date=None,
        top_ks=TOP_KS,
        tail_ks=TAIL_KS,
        twap_tag=BT_PRICE_TAG,
        run_ic=False,
    )

score_ens_oos = None
if score_ensemble_full is not None:
    d_oos = pd.to_datetime(score_ensemble_full.index).normalize() >= oos_start
    score_ens_oos = score_ensemble_full.loc[d_oos]

quick_oos_only("xgb", score_xgb_oos)
quick_oos_only("transformer", score_tf_oos)
quick_oos_only("ensemble", score_ens_oos)

INFO:Strategy.backtest.quick_backtest:分层回测完成: 371 交易日, 20 组, topK=(20, 50, 100), tailK=(20, 50, 100)
INFO:Strategy.backtest.quick_backtest:stock pool: score=5120 tradeable=3033 no_twap=0 limit_up=15 new=3 delist=0 st=607 prefix=1460
INFO:Strategy.backtest.quick_backtest:  group1: Ann=48.51% MDD=-28.49% SR=1.56 WR=56.33%
INFO:Strategy.backtest.quick_backtest:  group2: Ann=71.06% MDD=-17.16% SR=2.64 WR=60.92%
INFO:Strategy.backtest.quick_backtest:  group3: Ann=68.45% MDD=-17.08% SR=2.64 WR=62.53%
INFO:Strategy.backtest.quick_backtest:  group4: Ann=65.88% MDD=-15.95% SR=2.55 WR=60.65%
INFO:Strategy.backtest.quick_backtest:  group5: Ann=65.12% MDD=-15.33% SR=2.59 WR=60.92%
INFO:Strategy.backtest.quick_backtest:  group6: Ann=63.05% MDD=-15.33% SR=2.55 WR=61.46%
INFO:Strategy.backtest.quick_backtest:  group7: Ann=61.90% MDD=-16.67% SR=2.46 WR=60.11%
INFO:Strategy.backtest.quick_backtest:  group8: Ann=60.26% MDD=-15.68% SR=2.44 WR=60.11%
INFO:Strategy.backtest.quick_backtest:  group9: Ann=60.

## 6. 精细化事件回测（各模型 `score_full`，VAL 起至数据末）

与 main 中注释的 `BacktestRunner` 用法一致；`split_date` 标出 OOS 起点。

In [10]:
# from Strategy.backtest.event_backtest import BacktestRunner

# def event_one(name: str, score_full: pd.DataFrame):
#     if score_full is None or len(score_full) == 0:
#         return
#     evd = COMPARE_ROOT / name / "event_val_to_end"
#     evd.mkdir(parents=True, exist_ok=True)
#     runner = BacktestRunner(
#         score_df=score_full,
#         top_n=50,
#         n_quantile_groups=config.N_QUANTILE_GROUPS,
#         rebalance_freq=1,
#         initial_capital=config.INITIAL_CAPITAL,
#         frictionless=False,
#         twap_tag=BT_PRICE_TAG,
#     )
#     res = runner.run(start_date=config.VAL_START, end_date=None)
#     res.plot(save_dir=evd, split_date=config.OOS_START)
#     res.save_details(evd)
#     print(f"事件回测已写入: {evd}")

# event_one("xgb", score_xgb_full)
# event_one("transformer", score_tf_full)
# event_one("ensemble", score_ensemble_full)